## Script de sauvegarde et chargement du modèle

In [ ]:

from keras.models import save_model, load_model

def save_and_load_model(model, model_path='imdb_sentiment_model.keras'):
    """Save and load model."""
    save_model(model, model_path)
    print(f"✓ Model saved: {model_path}")
    
    loaded_model = load_model(model_path)
    print("✓ Model loaded successfully!")
    
    return loaded_model


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from keras.datasets import imdb
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import RMSprop
from keras.regularizers import l2
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

def load_data(num_words=10000, maxlen=500):
    """Load and prepare IMDB data."""
    print("\n=== LOADING DATA ===")
    
    (x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=num_words)
    
    x_train = [seq[:maxlen] for seq in x_train]
    x_test = [seq[:maxlen] for seq in x_test]
    
    print(f"✓ Train: {len(x_train)} | Test: {len(x_test)}")
    print(f"✓ Sample: {x_train[0][:20]}")
    print(f"✓ Label: {'Positive' if y_train[0] == 1 else 'Negative'}")
    
    return x_train, y_train, x_test, y_test

def vectorize_data(sequences, dim=10000):
    """Convert sequences to binary matrix."""
    data = np.zeros((len(sequences), dim))
    for i, seq in enumerate(sequences):
        data[i, seq] = 1
    return data

def prep_data(x_train, y_train, x_test, y_test, num_words=10000):
    """Vectorize and split data."""
    print("\n=== VECTORIZING DATA ===")
    
    x_train_vec = vectorize_data(x_train, num_words)
    x_test_vec = vectorize_data(x_test, num_words)
    y_train_vec = np.asarray(y_train).astype('float32')
    y_test_vec = np.asarray(y_test).astype('float32')
    
    print(f"✓ Train shape: {x_train_vec.shape}")
    print(f"✓ Test shape: {x_test_vec.shape}")
    
    val_size = int(0.25 * len(x_train_vec))
    
    x_val, y_val = x_train_vec[:val_size], y_train_vec[:val_size]
    x_train_final = x_train_vec[val_size:]
    y_train_final = y_train_vec[val_size:]
    
    print(f"✓ Val: {len(x_val)} | Train: {len(x_train_final)}")
    
    return x_train_final, x_val, x_test_vec, y_train_final, y_val, y_test_vec

def build_model(input_dim=10000, dropout=0.5, l2_reg=0.001):
    """Build sentiment model."""
    print("\n=== MODEL ARCHITECTURE ===")
    
    model = Sequential([
        Dense(64, activation='relu', input_shape=(input_dim,), kernel_regularizer=l2(l2_reg)),
        Dropout(dropout),
        Dense(32, activation='relu', kernel_regularizer=l2(l2_reg)),
        Dropout(dropout),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer=RMSprop(learning_rate=0.001),
                  loss='binary_crossentropy', metrics=['accuracy'])
    
    model.summary()
    return model

def train_model(model, x_train, y_train, x_val, y_val, epochs=20, batch_sz=512):
    """Train model and return history."""
    print("\n=== TRAINING ===")
    
    hist = model.fit(
        x_train, y_train,
        epochs=epochs, batch_size=batch_sz,
        validation_data=(x_val, y_val), verbose=1
    )
    
    return hist

def plot_history(h, title=""):
    """Plot loss and accuracy."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    epochs = range(1, len(h.history['loss']) + 1)
    
    ax1.plot(epochs, h.history['loss'], 'b-', label='Train', linewidth=2)
    ax1.plot(epochs, h.history['val_loss'], 'r-', label='Val', linewidth=2)
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{title}Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, h.history['accuracy'], 'b-', label='Train', linewidth=2)
    ax2.plot(epochs, h.history['val_accuracy'], 'r-', label='Val', linewidth=2)
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy')
    ax2.set_title(f'{title}Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('imdb_history.png', dpi=300, bbox_inches='tight')
    plt.show()

def find_best_epoch(h):
    """Find best epoch and detect overfitting."""
    print("\n=== OVERFITTING ANALYSIS ===")
    
    val_loss = h.history['val_loss']
    val_acc = h.history['val_accuracy']
    train_loss = h.history['loss']
    
    best_epoch = np.argmin(val_loss) + 1
    
    print(f"✓ Best epoch: {best_epoch}")
    print(f"✓ Best val loss: {min(val_loss):.4f}")
    print(f"✓ Best val acc: {max(val_acc):.4f}")
    
    for i in range(1, len(val_loss)):
        if val_loss[i] > val_loss[i-1]:
            print(f"⚠ Overfitting starts at epoch {i+1}")
            break
    
    return best_epoch

def eval_model(model, x_test, y_test):
    """Evaluate model on test set."""
    print("\n=== EVALUATION ===")
    
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    
    print(f"✓ Test Loss: {loss:.4f}")
    print(f"✓ Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    
    return loss, acc

def analyze_preds(model, x_test, y_test, n_samples=10):
    """Analyze predictions on samples."""
    print("\n=== PREDICTIONS ===")
    
    preds = model.predict(x_test[:n_samples], verbose=0)
    
    for i in range(n_samples):
        pred_class = 1 if preds[i] >= 0.5 else 0
        true_class = y_test[i]
        conf = preds[i][0] if pred_class == 1 else 1 - preds[i][0]
        
        status = "✓" if pred_class == true_class else "✗"
        sentiment = "Positive" if pred_class == 1 else "Negative"
        
        print(f"{status} Sample {i+1}: Pred={sentiment} ({conf:.0%}), "
              f"True={'Pos' if true_class == 1 else 'Neg'}")

def retrain_optimal(x_train, y_train, x_val, y_val, best_epoch):
    """Retrain with optimal epochs."""
    print(f"\n=== RETRAINING ({best_epoch} EPOCHS) ===")
    
    opt_model = build_model()
    h_opt = opt_model.fit(
        x_train, y_train,
        epochs=best_epoch, batch_size=512,
        validation_data=(x_val, y_val), verbose=1
    )
    
    return opt_model, h_opt

def main():
    """Main pipeline."""
    # 1. Load data
    x_train, y_train, x_test, y_test = load_data()
    
    # 2. Prepare data
    x_train, x_val, x_test, y_train, y_val, y_test = prep_data(
        x_train, y_train, x_test, y_test
    )
    
    # 3. Build model
    model = build_model()
    
    # 4. Train
    h = train_model(model, x_train, y_train, x_val, y_val, epochs=20)
    
    # 5. Visualize
    plot_history(h, "Initial - ")
    
    # 6. Find best epoch
    best_ep = find_best_epoch(h)
    
    # 7. Retrain optimal
    opt_model, h_opt = retrain_optimal(x_train, y_train, x_val, y_val, best_ep)
    
    # 8. Plot optimal
    plot_history(h_opt, "Optimal - ")
    
    # 9. Evaluate
    loss, acc = eval_model(opt_model, x_test, y_test)
    
    # 10. Analyze
    analyze_preds(opt_model, x_test, y_test)
    
    # 11. Summary
    print("\n=== SUMMARY ===")
    print(f"✓ Model trained successfully!")
    print(f"✓ Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(f"✓ Test Loss: {loss:.4f}")
    print(f"✓ Optimal epochs: {best_ep}")
    print(f"✓ History saved: imdb_history.png")
    print("="*50)

if __name__ == "__main__":
    main()


In [ ]:

def predict_sentiment(text, model, word_idx, num_words=10000, maxlen=500):
    """Predict sentiment of new text."""
    words = text.lower().split()
    seq = []
    
    for w in words:
        if w in word_idx and word_idx[w] < num_words:
            seq.append(word_idx[w])
    
    seq = seq[:maxlen]
    vec = vectorize_data([seq], num_words)
    pred = model.predict(vec)[0][0]
    
    sentiment = "POSITIVE" if pred >= 0.5 else "NEGATIVE"
    conf = pred if pred >= 0.5 else 1 - pred
    
    print(f"\nText: {text}")
    print(f"Sentiment: {sentiment} ({conf:.0%} confidence)")
    
    return pred, sentiment

# Usage example:
# word_idx = keras.datasets.imdb.get_word_index()
# predict_sentiment("This movie was fantastic!", model, word_idx)
